Computational Carpentry Part A: in this part a csv file containing all elements with the corresponding chemical and physical properties was analized to implement a function that calculates the molecular mass given a molecule.

In [ ]:
# Import of the pandas module to read the csv file. 
import pandas as pd
import re
import io
# The following code was used to handle a syntax problem, the first line of the file cotaining the names of the columns was read a single line due to the presence of double quotation marks
with open("periodic_table.csv", encoding="utf-8-sig") as f: 
    text = f.read()
text = re.sub(r'^"|"$', '', text, flags=re.MULTILINE)
text = text.replace('""', '"')
df = pd.read_csv(io.StringIO(text))
print(df.head())


   AtomicNumber Symbol       Name  AtomicMass CPKHexColor  \
0             1      H   Hydrogen    1.008000      FFFFFF   
1             2     He     Helium    4.002600      D9FFFF   
2             3     Li    Lithium    7.000000      CC80FF   
3             4     Be  Beryllium    9.012183      C2FF00   
4             5      B      Boron   10.810000      FFB5B5   

  ElectronConfiguration  Electronegativity  AtomicRadius  IonizationEnergy  \
0                   1s1               2.20         120.0            13.598   
1                   1s2                NaN         140.0            24.587   
2               [He]2s1               0.98         182.0             5.392   
3               [He]2s2               1.57         153.0             9.323   
4           [He]2s2 2p1               2.04         192.0             8.298   

   ElectronAffinity OxidationStates StandardState  MeltingPoint  BoilingPoint  \
0             0.754          +1, -1           Gas         13.81         20.28   
1 

In [13]:
#Creation of a dictionary where the keys are the atomic symbols while the values are the atomic masses
atoms_to_mass = dict(zip(df["Symbol"], df["AtomicMass"]))
print(atoms_to_mass)

{'H': 1.008, 'He': 4.0026, 'Li': 7.0, 'Be': 9.012183, 'B': 10.81, 'C': 12.011, 'N': 14.007, 'O': 15.999, 'F': 18.99840316, 'Ne': 20.18, 'Na': 22.9897693, 'Mg': 24.305, 'Al': 26.981538, 'Si': 28.085, 'P': 30.973762, 'S': 32.07, 'Cl': 35.45, 'Ar': 39.9, 'K': 39.0983, 'Ca': 40.08, 'Sc': 44.95591, 'Ti': 47.867, 'V': 50.9415, 'Cr': 51.996, 'Mn': 54.93804, 'Fe': 55.84, 'Co': 58.93319, 'Ni': 58.693, 'Cu': 63.55, 'Zn': 65.4, 'Ga': 69.723, 'Ge': 72.63, 'As': 74.92159, 'Se': 78.97, 'Br': 79.9, 'Kr': 83.8, 'Rb': 85.468, 'Sr': 87.62, 'Y': 88.90584, 'Zr': 91.22, 'Nb': 92.90637, 'Mo': 95.95, 'Tc': 96.90636, 'Ru': 101.1, 'Rh': 102.9055, 'Pd': 106.42, 'Ag': 107.868, 'Cd': 112.41, 'In': 114.818, 'Sn': 118.71, 'Sb': 121.76, 'Te': 127.6, 'I': 126.9045, 'Xe': 131.29, 'Cs': 132.905452, 'Ba': 137.33, 'La': 138.9055, 'Ce': 140.116, 'Pr': 140.90766, 'Nd': 144.24, 'Pm': 144.91276, 'Sm': 150.4, 'Eu': 151.964, 'Gd': 157.2, 'Tb': 158.92535, 'Dy': 162.5, 'Ho': 164.93033, 'Er': 167.26, 'Tm': 168.93422, 'Yb': 173.05

In [ ]:
#Implementation of a function that given a molecular formula returns the molecular weight of the molecule
def molecular_weight_calculator(molecule:str):
    molecular_weight = 0                           #Setting of the initial variables
    i = 0
    n = len(molecule)

    while i < n :
        starting_element = i                      #Definition of an element by considering its starting letter
        i +=1
        if i < n and molecule[i].islower():       #Checks if the following character in the string is a lowercase letter
            i +=1
        element = molecule[starting_element:i]    #Definition of the element
        starting_number = i                       #Defines the first number of the coefficient
        while i < n and molecule[i].isdigit():    #Determination of the whole coefficient
            i += 1
        coefficient = molecule[starting_number:i]
        numerical_coefficient = int(coefficient) if coefficient else 1  #Conversion of the coefficient from a string value to an integer

        if element in atoms_to_mass:                 #Determination of the molecular mass by considering if an element in the molecule given is in the dictionary previously created
            molecular_weight += numerical_coefficient * atoms_to_mass[element]
    return molecular_weight

    
print(molecular_weight_calculator("H2O"))
print(molecular_weight_calculator("C6H12O6"))
print(molecular_weight_calculator("N2O7H30"))

18.015
180.156
170.247


In [39]:
#Implementation of a function that given a molecular formula returns the molecular weight of the molecule handling molecular formulas with parentheses
def molecular_weight_calculator_parenthesis(molecule:str):
    molecular_weight = 0                           #Setting of the initial variables
    i = 0
    n = len(molecule)   
    while i < n:
        if molecule[i] == "(":                     
            parenthesis_count = 1              #Setting of a counter to keep track of parentheses
            next_to_parenthesis = i + 1        
            i += 1
            while i < n and parenthesis_count > 0:  #Calculating the parentheses count that is 0 only if a same number of opening and closing parentheses are found
                if molecule[i] == "(":
                    parenthesis_count +=1
                elif molecule[i] == ")":
                    parenthesis_count -=1
                i +=1
            inside_weight = molecular_weight_calculator_parenthesis(molecule[next_to_parenthesis:i-1]) #Calculates the weight of the molecular fragment inside the parentheses
            after_closing_parenthesis= i 
            while i < n and molecule[i].isdigit():   
                i +=1 
            coefficient = molecule[after_closing_parenthesis:i]   #Determines the coefficient outside the parentheses
            numerical_coefficient = int(molecule[after_closing_parenthesis:i]) if after_closing_parenthesis < i else 1 #Converts the string coefficient to an integer value and sets it to one if the next character after the parenthesis is a letter
            molecular_weight += numerical_coefficient * inside_weight    #Calculates the molecular weight of the fragment inside the parentheses

        elif molecule[i].isupper():
            starting_element = i                      #Definition of an element by considering its starting letter
            i +=1
            if i < n and molecule[i].islower():       #Checks if the following character in the string is a lowercase letter
                i +=1
            element = molecule[starting_element:i]    #Definition of the element
            starting_number = i                       #Defines the first number of the coefficient
            while i < n and molecule[i].isdigit():    #Determination of the whole coefficient
                    i += 1
            coefficient = molecule[starting_number:i]
            numerical_coefficient = int(coefficient) if coefficient else 1  #Conversion of the coefficient from a string value to an integer
            
            if element in atoms_to_mass:                 #Determination of the molecular mass by considering if an element in the molecule given is in the dictionary previously created
                molecular_weight += numerical_coefficient * atoms_to_mass[element]
        else:
            i+=1
    return molecular_weight

print(molecular_weight_calculator_parenthesis("Ca(OH)2"))
print(molecular_weight_calculator_parenthesis("CH4"))     
print(molecular_weight_calculator_parenthesis("Al2(CO3)3"))
        

        



74.094
16.043
233.987076


In [46]:
#Implementation of a function that given a molecular formula returns the molecular weight of the molecule handling molecular formulas with parentheses and coordination hydrates
#The function treats the molecule as separate parts divised by the "."
def molecular_weight_calculator_parenthesis_coordination(molecule:str):
    molecular_weight = 0                           #Setting of the initial variables
    i = 0
    n = len(molecule)   
    while i < n:
        if molecule[i] == ".":        
            i +=1
            after_point = i 
            while i < n and molecule[i].isdigit():    #Determining the coefficient of the hydrate
                i +=1
            hydrate_coeff = int(molecule[after_point:i]) if after_point < i else 1  #Turning the coefficient from a string to an integer
            hydrate_formula = molecule[i:]                                          #Finding the molecular formula of the hydrate
            return molecular_weight + (hydrate_coeff * molecular_weight_calculator_parenthesis_coordination(hydrate_formula)) #Calculating the total weight of the molecule


        elif molecule[i] == "(":                     
            parenthesis_count = 1              #Setting of a counter to keep track of parentheses
            next_to_parenthesis = i + 1        
            i += 1
            while i < n and parenthesis_count > 0:  #Calculating the parentheses count that is 0 only if a same number of opening and closing parentheses are found
                if molecule[i] == "(":
                    parenthesis_count +=1
                elif molecule[i] == ")":
                    parenthesis_count -=1
                i +=1
            inside_weight = molecular_weight_calculator_parenthesis_coordination(molecule[next_to_parenthesis:i-1]) #Calculates the weight of the molecular fragment inside the parentheses
            after_closing_parenthesis= i 
            while i < n and molecule[i].isdigit():   
                i +=1 
            coefficient = molecule[after_closing_parenthesis:i]   #Determines the coefficient outside the parentheses
            numerical_coefficient = int(molecule[after_closing_parenthesis:i]) if after_closing_parenthesis < i else 1 #Converts the string coefficient to an integer value and sets it to one if the next character after the parenthesis is a letter
            molecular_weight += numerical_coefficient * inside_weight    #Calculates the molecular weight of the fragment inside the parentheses

        elif molecule[i].isupper():
            starting_element = i                      #Definition of an element by considering its starting letter
            i +=1
            if i < n and molecule[i].islower():       #Checks if the following character in the string is a lowercase letter
                i +=1
            element = molecule[starting_element:i]    #Definition of the element
            starting_number = i                       #Defines the first number of the coefficient
            while i < n and molecule[i].isdigit():    #Determination of the whole coefficient
                    i += 1
            coefficient = molecule[starting_number:i]
            numerical_coefficient = int(coefficient) if coefficient else 1  #Conversion of the coefficient from a string value to an integer
            
            if element in atoms_to_mass:                 #Determination of the molecular mass by considering if an element in the molecule given is in the dictionary previously created
                molecular_weight += numerical_coefficient * atoms_to_mass[element]
        else:
            i+=1
    return molecular_weight

print(molecular_weight_calculator_parenthesis_coordination("CuSO4.5H2O"))
print(molecular_weight_calculator_parenthesis_coordination("Na2B4O7.10H2O"))
print(molecular_weight_calculator_parenthesis_coordination("Na2(B4O7).10H2O"))

249.69100000000003
381.3625386
381.3625386


Failed Experiments

In [ ]:
#Implementation of a function that given a molecular formula returns the molecular weight of the molecule
def molecular_weight_calculator(molecule:str):
    molecular_weight = 0

    for element in atoms_to_mass:
        if element in molecule:
            coefficient_position = molecule.index(element) + len(element)

            if coefficient_position < len(molecule) and molecule[coefficient_position].isdigit():
                coefficient_value = int(molecule[coefficient_position])
            else:
                coefficient_value = 1

            molecular_weight += coefficient_value* atoms_to_mass[element]
    return molecular_weight
print(molecular_weight_calculator("H2O"))
print(molecular_weight_calculator("C6H12O6"))
#The function works for H2O but not for C6H12O6 as the coefficient_position finds the position just of the fist digit of the number, therefore only one hydrogen atom is considered and not 12.

18.015
169.06799999999998
